# TV-03 -- Internalisation du raisonnement (CoT -> calcul interne) -- v2 multi-seed

Troisieme tranche d'execution de l'Epic #17540 (Russell & Norvig arc B, raisonnement internalise). Cette v2 livre :

1. Le module tv/task.py etendu : nouvelle fonction entrainer_multi_seed() qui satisfait le protocole PR review-discipline §C (>=4 graines parmi 0/1/7/42/99).
2. Mesure multi-seed (4 graines) sur la tache multi-sauts 3 sauts : moyenne et ecart-type sur l'exactitude, la perplexite et le temps d'entrainement. Verification du discriminant H.2 sur la base solide multi-seed.
3. Validation cross-grain : single-hop reste triviale a 100 %, multi-sauts discriminee, single+multi-sauts >=4 graines -- protocole §C strict respecte.

## Pourquoi cette v2

La v1 (PR #17697 v1, c.808) posait le discriminant H.2 sur 1 graine. Tell c.1493 strict fondateur nuance -- la mesure multi-seed est le complement attendu par le protocole PR review-discipline §C et par la qualification organ-first (grain DEEP multi-seed >=4). Cette v2 livre cette mesure.

## Ce qui n'est PAS dans cette tranche

- Variante CoT supervisee (squelette conserve v1 cellule 9, non execute).
- Lecture SAE des representations internes (autre lane po-2027:CoursIA).
- Comparaison CoT vs answer-only (grain suivant du programme).


In [1]:
import math
import sys
import time

import torch
import torch.nn.functional as F

sys.path.insert(0, '.')
from tv import (  # noqa: E402
    PetitLM, Vocab,
    evaluer_single_hop, evaluer_multi_hop,
    entrainer, entrainer_multi_seed,
)

print(f'torch {torch.__version__} | python {sys.version_info.major}.{sys.version_info.minor}')
print('package tv charge OK : model + task (single-hop + multi-hop + multi-seed) disponibles')


torch 2.13.0+cu126 | python 3.13
package tv charge OK : model + task (single-hop + multi-hop + multi-seed) disponibles


## 1. Vocabulaires

Deux vocabulaires de même structure :

- **Single-hop** : `N_MARQUEURS=8`, `N_REMPLISSAGE=10`, `N_QUESTIONS=1` (le `QUESTION(0)` trivial). Vocab total 20 tokens.
- **Multi-sauts** : `N_MARQUEURS=8`, `N_REMPLISSAGE=10`, `N_QUESTIONS=3` (la cible est le `q`-ième marqueur, avec `q` choisi uniformément sur [0, 3)). Vocab total 22 tokens.

Hasard exactitude : 1/8 dans les deux cas (la cible est l'un des 8 marqueurs ; le mécanisme à apprendre est la sélection conditionnelle, pas la mémorisation).

In [2]:
T = 64  # longueur de sequence commune aux deux taches
v_single = Vocab(N_MARQUEURS=8, N_REMPLISSAGE=10, N_QUESTIONS=1)
v_multi = Vocab(N_MARQUEURS=8, N_REMPLISSAGE=10, N_QUESTIONS=3)

def fabrique(vocab):
    """Fabrique un MHA vierge (d_model=64, 4 tetes, 2 couches, fenetre pleine)."""
    return PetitLM(
        vocab=vocab.VOCAB,
        d_model=64,
        n_heads=4,
        n_kv_heads=4,
        window=None,
        n_couches=2,
    )

print(f'Vocab single-hop : VOCAB={v_single.VOCAB}, N_QUESTIONS={v_single.N_QUESTIONS}')
print(f'Vocab multi-sauts : VOCAB={v_multi.VOCAB}, N_QUESTIONS={v_multi.N_QUESTIONS}')
print(f'Hasard exactitude : 1 / {v_single.N_MARQUEURS} = {1/v_single.N_MARQUEURS:.4f}')


Vocab single-hop : VOCAB=20, N_QUESTIONS=1
Vocab multi-sauts : VOCAB=22, N_QUESTIONS=3
Hasard exactitude : 1 / 8 = 0.1250


## 2. Mesure multi-seed single-hop (4 graines, 300 pas)

Single-hop est trivialement resolue par MHA 102K params : on s'attend a 1.0000 +/- ~0 sur 4 graines.


In [3]:
GRAINES = [0, 1, 7, 42]
PAS = 300

result_s = entrainer_multi_seed(
    fabrique, v_single, T=T, multi_hop=False, graines=GRAINES, pas=PAS
)
print(f'SINGLE-HOP multi-seed ({len(GRAINES)} graines, {PAS} pas) :')
print(f'  EXACTITUDE = {result_s["acc_moy"]:.4f} +/- {result_s["acc_std"]:.4f}  (hasard = 0.1250)')
print(f'  PERPLEXITE = {result_s["ppl_moy"]:.4f} +/- {result_s["ppl_std"]:.4f}  (hasard = 8.0)')
print(f'  Secondes total = {result_s["secondes"]:.2f} s')
for graine, acc, ppl, sec in result_s["brut"]:
    print(f'    graine {graine} : acc={acc:.4f} ppl={ppl:.4f} sec={sec:.2f}')


SINGLE-HOP multi-seed (4 graines, 300 pas) :
  EXACTITUDE = 1.0000 +/- 0.0000  (hasard = 0.1250)
  PERPLEXITE = 1.0015 +/- 0.0000  (hasard = 8.0)
  Secondes total = 74.72 s
    graine 0 : acc=1.0000 ppl=1.0015 sec=20.83
    graine 1 : acc=1.0000 ppl=1.0015 sec=17.14
    graine 7 : acc=1.0000 ppl=1.0015 sec=16.05
    graine 42 : acc=1.0000 ppl=1.0015 sec=17.58


## 3. Mesure multi-seed multi-sauts (4 graines, 300 pas)

Multi-sauts 3 questions : le discriminant H.2 doit tenir sur 4 graines (moyenne significativement au-dessus du hasard, sans atteindre 1.0).


In [4]:
result_m = entrainer_multi_seed(
    fabrique, v_multi, T=T, multi_hop=True, graines=GRAINES, pas=PAS
)
print(f'MULTI-SAUTS multi-seed ({len(GRAINES)} graines, {PAS} pas, 3 sauts) :')
print(f'  EXACTITUDE = {result_m["acc_moy"]:.4f} +/- {result_m["acc_std"]:.4f}  (hasard = 0.1250)')
print(f'  PERPLEXITE = {result_m["ppl_moy"]:.4f} +/- {result_m["ppl_std"]:.4f}  (hasard = 8.0)')
print(f'  Secondes total = {result_m["secondes"]:.2f} s')
for graine, acc, ppl, sec in result_m["brut"]:
    print(f'    graine {graine} : acc={acc:.4f} ppl={ppl:.4f} sec={sec:.2f}')


MULTI-SAUTS multi-seed (4 graines, 300 pas, 3 sauts) :
  EXACTITUDE = 0.4019 +/- 0.0161  (hasard = 0.1250)
  PERPLEXITE = 2.9156 +/- 0.0943  (hasard = 8.0)
  Secondes total = 67.73 s
    graine 0 : acc=0.3945 ppl=2.9109 sec=16.53
    graine 1 : acc=0.4180 ppl=3.0707 sec=17.09
    graine 7 : acc=0.4160 ppl=2.8310 sec=17.75
    graine 42 : acc=0.3789 ppl=2.8499 sec=15.69


## 4. Bilan tranche v2

Multi-seed >=4 sur single-hop vs multi-sauts -- discriminant H.2 mesure sur 4 graines.


In [5]:
print('BILAN multi-seed :')
print(f'  single-hop : {result_s["acc_moy"]:.4f} +/- {result_s["acc_std"]:.4f}')
print(f'  multi-sauts : {result_m["acc_moy"]:.4f} +/- {result_m["acc_std"]:.4f}')
rapport = result_m["acc_moy"] / result_s["acc_moy"] if result_s["acc_moy"] > 0 else float("inf")
print(f'  rapport multi/single = {rapport:.3f}')
print(f'  secondes total (4 graines single + 4 graines multi) = {result_s["secondes"] + result_m["secondes"]:.2f} s')
print()
if result_s['acc_moy'] >= 0.99 and result_m['acc_moy'] < result_s['acc_moy']:
    print('Conclusion : Single-hop trivialement resolu (>=0.99) ; multi-sauts plus bas avec >=4 graines.')
    print('Discriminant H.2 multi-seed : OK (single >> multi, multi > hasard).')
else:
    print(f'ATTENTION : single-hop = {result_s["acc_moy"]:.4f}, multi-sauts = {result_m["acc_moy"]:.4f}.')
    print('Le discriminant H.2 n est PAS clairement tenu. Investiguer la graine fautive.')


BILAN multi-seed :
  single-hop : 1.0000 +/- 0.0000
  multi-sauts : 0.4019 +/- 0.0161
  rapport multi/single = 0.402
  secondes total (4 graines single + 4 graines multi) = 142.45 s

Conclusion : Single-hop trivialement resolu (>=0.99) ; multi-sauts plus bas avec >=4 graines.
Discriminant H.2 multi-seed : OK (single >> multi, multi > hasard).
